## P69_73

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from copy import deepcopy
import numpy as np
import matplotlib.pyplot as plt
import torch
import gymnasium as gym

import stable_pretraining as spt
import stable_worldmodel as swm
from torchvision.transforms import v2 as transforms
from sklearn.preprocessing import StandardScaler

import sys
sys.path.append('data/hackin/le-wm') #Path to cloned repo
from decoder import CLSDecoder

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

In [ ]:
from __future__ import annotations
import numpy as np
import scipy.ndimage as ndi
 
 
def _hex(h: str) -> np.ndarray:
    h = h.lstrip("#")
    return np.array([int(h[i:i+2], 16) for i in (0, 2, 4)], dtype=np.float32)
 
 
# Welch Labs palette
CHILL_BROWN = _hex("#948979")
YELLOW      = _hex("#ffd35a")
YELLOW_FADE = _hex("#7f6a2d")
BLUE        = _hex("#00AEEF")
BLACK       = np.zeros(3, dtype=np.float32)
 

In [ ]:
env = gym.make("swm/PushT-v1", render_mode="rgb_array")
obs, info = env.reset(seed=0)
print("action_space:", env.action_space)
print("frame:       ", env.render().shape, env.render().dtype)

In [ ]:
ds = swm.data.HDF5Dataset(name="pusht_expert_train",
                          keys_to_cache=["action", "state"])

# --- action scaler (same as notebook 10) ---
all_actions = ds.get_col_data("action")
if all_actions.ndim == 3:
    all_actions_2d = all_actions.reshape(-1, all_actions.shape[-1])
else:
    all_actions_2d = all_actions
all_actions_2d = all_actions_2d[~np.isnan(all_actions_2d).any(axis=1)]
action_scaler = StandardScaler().fit(all_actions_2d)
print("action mean:", action_scaler.mean_)
print("action std: ", action_scaler.scale_)

In [ ]:
img_transform = transforms.Compose([
    transforms.ToImage(),
    transforms.ToDtype(torch.float32, scale=True),
    transforms.Normalize(**spt.data.dataset_stats.ImageNet),
    transforms.Resize(size=224),
])

# JEPA encoder + predictor + action_encoder bundled together
cost_model = swm.policy.AutoCostModel("pusht/lewm").to(DEVICE).eval()
# cost_model = swm.policy.AutoCostModel("from_lucas/lewm").to(DEVICE).eval()
cost_model.requires_grad_(False)
cost_model.interpolate_pos_encoding = True

encoder = cost_model.encoder.eval()
for p in encoder.parameters():
    p.requires_grad_(False)

# CLS-token decoder (ViT-S/14 patches → 224x224 RGB)
# decoder_weights = torch.load("from_lucas/decoder.pt", map_location=DEVICE)
# decoder_weights = torch.load("decoder_mc_10_epochs.pt", map_location=DEVICE)

#Path to Matthew's trained decoder
state_dict = torch.load('data/hackin/le-wm/decoder_mc_10_epochs.pt', weights_only=True, map_location=torch.device('cpu'))
#decoder = CLSDecoder(cls_dim=192, img_size=224, patch_size=14,
#                     dim=256, heads=8, depth=4)
decoder_mc = CLSDecoder(cls_dim=192, img_size=224, patch_size=14,
                     dim=256, heads=8, depth=3)
decoder_mc.load_state_dict(state_dict, strict=False)
# decoder_mc.load_state_dict(decoder_weights)
decoder_mc = decoder_mc.to(DEVICE).eval()
decoder_mc.requires_grad_(False);

In [ ]:
import torch.nn as nn

enc_dim  = cost_model.encoder.config.hidden_size
proj_dim = 192

# Load the trained adapter model
adapter_state_dict = torch.load('data/hackin/le-wm/adapter_weights.pt', weights_only=True, map_location=torch.device('cpu'))

adapter = nn.Sequential(
    nn.Linear(proj_dim, 512),
    nn.GELU(),
    nn.Linear(512, enc_dim),
)

adapter.load_state_dict(adapter_state_dict, strict=False)
adapter = adapter.to(DEVICE).eval()
adapter.requires_grad_(False);

In [ ]:
@torch.no_grad()
def encode_frame(rgb_uint8, cost_model, transform, device):
    """(H, W, 3) uint8 -> (1, D) projected emb."""
    px = transform(rgb_uint8).unsqueeze(0).unsqueeze(0).to(device)  # (1, 1, 3, H, W)
    info = {"pixels": px}
    info = cost_model.encode(info)
    return info["emb"][:, 0]   # (1, T=1, D) -> (1, D)


@torch.no_grad()
def decode_emb(emb_1d, adapter, decoder):
    """(1, D) projected emb -> (H, W, 3) float [0,1] for imshow."""
    cls_hat = adapter(emb_1d)                              # (1, enc_dim)
    img = decoder(cls_hat)[0].detach().permute(1, 2, 0).cpu().numpy()
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    return img


def pack_action_block(actions_5x2, action_scaler, device):
    """(5, 2) env-scale actions -> (1, 10) standardized tensor."""
    a = np.asarray(actions_5x2, dtype=np.float32)
    assert a.shape == (5, 2), f"expected (5, 2), got {a.shape}"
    a_std = action_scaler.transform(a).astype(np.float32)  # (5, 2)
    a_flat = a_std.reshape(-1)                             # (10,)
    return torch.from_numpy(a_flat).to(device).unsqueeze(0)  # (1, 10)


@torch.no_grad()
def predict_next(cost_model, emb_hist, act_hist, history_size=3):
    """
    emb_hist: list of (1, D) tensors — projected embeddings.
    act_hist: list of (1, 10) tensors — packed action blocks.
    Returns: (1, D) next projected embedding.
    """
    assert len(emb_hist) == len(act_hist), (
        f"history mismatch: {len(emb_hist)} embs vs {len(act_hist)} acts"
    )
    HS = history_size
    emb_seq = torch.stack(emb_hist[-HS:], dim=1)    # (1, L, D)
    act_seq = torch.stack(act_hist[-HS:], dim=1)    # (1, L, 10)
    act_emb = cost_model.action_encoder(act_seq)    # (1, L, A_emb)
    pred    = cost_model.predict(emb_seq, act_emb)  # (1, L, D)
    return pred[:, -1]                              # (1, D)


@torch.no_grad()
def play_game_v2(actions_5x2, cost_model, decoder, adapter, scaler,
              act_hist, emb_hist, frames, titles, device):
    # 1) pack action — store as (1, 10)
    a_block = pack_action_block(actions_5x2, scaler, device)  # (1, 10)
    act_hist.append(a_block)

    # 2) predict next projected embedding
    next_emb = predict_next(cost_model, emb_hist, act_hist, history_size=3)  # (1, D)
    emb_hist.append(next_emb)

    # 3) adapter -> decoder
    img = decode_emb(next_emb, adapter, decoder)
    frames.append(img)
    titles.append(f"t={(len(frames)-1)*5} (decoded)")

    show_strip(frames, titles)
    return next_emb, act_hist, emb_hist, frames, titles


def show_strip(frames, titles=None, figsize_per=2.2):
    """Plot a horizontal strip of frames."""
    n = len(frames)
    fig, axes = plt.subplots(1, n, figsize=(figsize_per * n, figsize_per))
    if n == 1:
        axes = [axes]
    for i, ax in enumerate(axes):
        ax.imshow(frames[i])
        ax.axis("off")
        if titles is not None:
            ax.set_title(titles[i], fontsize=9)
    plt.tight_layout()
    plt.show()


def show_images(original, recon, titles=None, figsize_per=2.2):
    fig, axes = plt.subplots(1, 2, figsize=(figsize_per * 2, figsize_per))
    
    # Display images on specific axes
    axes[0].imshow(original)
    axes[0].set_title("Original")

    axes[1].imshow(recon)
    axes[1].set_title("Reconstructed")
    
    plt.tight_layout()
    plt.show()

In [ ]:
DEFAULT_PALETTE: dict = {
    # ── source anchors (sampled by hand from the PushT renderer) ─────────
    "src_bg":            np.array([254, 254, 254], dtype=np.float32),
    "src_frame_border":  np.array([197,	197,	197		], dtype=np.float32),
    "src_target_fill":   np.array([91,	229,	104	], dtype=np.float32),  # green T
    "src_movable_fill":  np.array([92,	121,	147	], dtype=np.float32),  # blue-gray T
    "src_effector_fill": np.array([11,	78,	248	], dtype=np.float32),  # blue dot
    "src_target_outline":   np.array([104,	231,	115	], dtype=np.float32),  # green T
    "src_movable_outline":  np.array([64,	85,	106	], dtype=np.float32),  # blue-gray T
    "src_effector_outline": np.array([11,	65,	212	], dtype=np.float32),  # blue dot
    
    # ── target colors ────────────────────────────────────────────────────
    "tgt_bg":               BLACK,
    "tgt_frame_border":     BLACK,
    "tgt_target_fill":      CHILL_BROWN,
    "tgt_target_outline":   CHILL_BROWN, # * 0.55,
    "tgt_movable_fill":     BLUE,
    "tgt_movable_outline":  BLUE * 0.75,
    "tgt_effector_fill":    YELLOW,
    "tgt_effector_outline": YELLOW,
 
    # ── fill-mask thresholds (loose color-ball radii) ────────────────────
    # Only used to seed the Voronoi label propagation; doesn't need to be tight.
    "r_target_fill":   45.0,
    "r_movable_fill":  22.0,
    "r_effector_fill": 55.0,
 
    # ── connected-component cleanup ──────────────────────────────────────
    # Drop fill-mask components smaller than this many pixels (kills stray AA
    # pixels that happen to fall inside a fill color ball).
    "min_component_size": 10,
}
 


DECODED_PALETTE: dict = {
    # ── source anchors (sampled by hand from the PushT renderer) ─────────
    "src_bg":            np.array([203,	214,	226	], dtype=np.float32),
    "src_frame_border":  np.array([156,	164,	178	], dtype=np.float32),
    # "src_target_fill":   np.array([64,	184,	93	], dtype=np.float32),  # green T
    "src_target_fill":   np.array([88,	201,	112	], dtype=np.float32),  # green T
    
    # "src_movable_fill":  np.array([69,	100,	137	], dtype=np.float32),  # blue-gray T
    "src_movable_fill":  np.array([92,	121,	147	], dtype=np.float32),  # blue-gray T

    "src_effector_fill": np.array([ 15,	69,	215	], dtype=np.float32),  # blue dot
    "src_target_outline":   np.array([87,	172,	115	], dtype=np.float32),  # green T
    "src_movable_outline":  np.array([53,	77,	103	], dtype=np.float32),  # blue-gray T
    "src_effector_outline": np.array([14,	61,	175	], dtype=np.float32),  # blue dot
 
    # ── target colors ────────────────────────────────────────────────────
    "tgt_bg":               BLACK,
    "tgt_frame_border":     BLACK,
    "tgt_target_fill":      CHILL_BROWN,
    "tgt_target_outline":   CHILL_BROWN, # * 0.55,
    "tgt_movable_fill":     BLUE,
    "tgt_movable_outline":  BLUE * 0.75,
    "tgt_effector_fill":    YELLOW,
    "tgt_effector_outline": YELLOW,
 
    # ── fill-mask thresholds (loose color-ball radii) ────────────────────
    # Only used to seed the Voronoi label propagation; doesn't need to be tight.
    "r_target_fill":   45.0,
    "r_movable_fill":  22.0,
    "r_effector_fill": 55.0,
 
    # ── connected-component cleanup ──────────────────────────────────────
    # Drop fill-mask components smaller than this many pixels (kills stray AA
    # pixels that happen to fall inside a fill color ball).
    "min_component_size": 10,
}
 
 
# Per-shape (label_id, fill_src_key, outline_src_key, fill_tgt_key, outline_tgt_key)
_SHAPES = [
    (1, "src_target_fill",   "src_target_outline",   "tgt_target_fill",   "tgt_target_outline"),
    (2, "src_movable_fill",  "src_movable_outline",  "tgt_movable_fill",  "tgt_movable_outline"),
    (3, "src_effector_fill", "src_effector_outline", "tgt_effector_fill", "tgt_effector_outline"),
]
 
 
def _keep_largest_components(mask: np.ndarray, min_size: int) -> np.ndarray:
    """Drop connected components smaller than min_size pixels."""
    lab, n = ndi.label(mask)
    if n == 0:
        return mask
    sizes = ndi.sum_labels(mask, lab, range(1, n + 1))
    keep = np.zeros(n + 1, dtype=bool)
    keep[1:] = sizes >= min_size
    return keep[lab]
 
 
def _segment_shapes(img: np.ndarray, p: dict) -> np.ndarray:
    """Per-pixel shape label via threshold + cleanup + Voronoi propagation."""
    def ball(anchor, r):
        return np.linalg.norm(img - anchor, axis=-1) < r
 
    min_sz = int(p["min_component_size"])
    labels = np.zeros(img.shape[:2], dtype=np.int32)
    labels[_keep_largest_components(ball(p["src_target_fill"],   p["r_target_fill"]),   min_sz)] = 1
    labels[_keep_largest_components(ball(p["src_movable_fill"],  p["r_movable_fill"]),  min_sz)] = 2
    labels[_keep_largest_components(ball(p["src_effector_fill"], p["r_effector_fill"]), min_sz)] = 3
 
    if (labels == 0).all():
        # Pure-background frame; return zeros — output will be BLACK everywhere via bg anchor.
        return labels
 
    _, idx = ndi.distance_transform_edt(labels == 0, return_indices=True)
    return labels[idx[0], idx[1]]
 
 
def recolor_pusht(
    frame: np.ndarray,
    palette: dict | None = None,
    sigma: float = 22.0,
) -> np.ndarray:
    """Recolor one PushT frame.
 
    Args:
        frame:   (H, W, 3) uint8 RGB array.
        palette: dict matching DEFAULT_PALETTE structure (or None for defaults).
        sigma:   Gaussian σ in RGB units. 12 = very crisp, 22 = default, 35 = soft.
 
    Returns:
        (H, W, 3) uint8 RGB array.
    """
    p = palette if palette is not None else DEFAULT_PALETTE
    img = frame.astype(np.float32)
    H, W, _ = img.shape
 
    # 1. Spatial label per pixel
    lbl = _segment_shapes(img, p)
 
    # 2. Per-pixel fill/outline anchors based on shape label
    src_fill_pp    = np.empty_like(img)
    src_outline_pp = np.empty_like(img)
    tgt_fill_pp    = np.empty_like(img)
    tgt_outline_pp = np.empty_like(img)
    for lbl_val, sf_key, so_key, tf_key, to_key in _SHAPES:
        m = (lbl == lbl_val)
        if not m.any():
            continue
        src_fill_pp[m]    = p[sf_key]
        src_outline_pp[m] = p[so_key]
        tgt_fill_pp[m]    = p[tf_key]
        tgt_outline_pp[m] = p[to_key]
 
    # Fallback: if no shape was detected anywhere, point the shape anchors at bg
    # so they don't contribute spurious color via uninitialized memory.
    if (lbl == 0).any():
        src_fill_pp[lbl == 0]    = p["src_bg"]
        src_outline_pp[lbl == 0] = p["src_bg"]
        tgt_fill_pp[lbl == 0]    = p["tgt_bg"]
        tgt_outline_pp[lbl == 0] = p["tgt_bg"]
 
    # 3. Global bg + frame_border anchors broadcast to per-pixel shape
    bg_src    = np.broadcast_to(p["src_bg"],           img.shape)
    bg_tgt    = np.broadcast_to(p["tgt_bg"],           img.shape)
    frame_src = np.broadcast_to(p["src_frame_border"], img.shape)
    frame_tgt = np.broadcast_to(p["tgt_frame_border"], img.shape)
 
    src_pp = np.stack([src_fill_pp, src_outline_pp, bg_src, frame_src], axis=-2)  # (H, W, 4, 3)
    tgt_pp = np.stack([tgt_fill_pp, tgt_outline_pp, bg_tgt, frame_tgt], axis=-2)
 
    # 4. Gaussian soft blend over the 4 anchors
    d2 = ((img[..., None, :] - src_pp) ** 2).sum(-1)            # (H, W, 4)
    w  = np.exp(-d2 / (2.0 * sigma * sigma))
    w /= w.sum(-1, keepdims=True)
    out = (w[..., None] * tgt_pp).sum(-2)                        # (H, W, 3)
 
    return np.clip(out, 0, 255).astype(np.uint8)

## Play Video Game

In [ ]:
from PIL import Image

save_dir='data/hackin/p69'

## Move Left

In [ ]:
# fresh env just to get a deterministic starting frame matched to Part A's seed=0 reset.
env_game = gym.make("swm/PushT-v1", render_mode="rgb_array")
env_game.reset(seed=2)
start_frame = env_game.render()
env_game.close()

# Initialise histories with the first observed frame
first_emb = encode_frame(start_frame, cost_model=cost_model, transform=img_transform, device=DEVICE)  # (1, D)
emb_hist = [first_emb]
act_hist = []

# Seed frames with the real observation for visual reference
frames = [((start_frame - start_frame.min()) / (start_frame.max() - start_frame.min() + 1e-8))]
titles = ["t=0 (real)"]

show_strip(frames, titles)

In [ ]:
img_r=recolor_pusht(np.round(frames[0]*255).astype('uint8'), palette=DEFAULT_PALETTE, sigma=45)

Image.fromarray(img_r).save(save_dir+'/start.png')

In [ ]:
Image.fromarray(img_r)

In [ ]:
actions = np.array([
    [-0.2, 0.0],
    [-0.2, 0.0],
    [-0.2, 0.0],
    [-0.2, 0.0],
    [-0.2, 0.0], #LEFT
], dtype=np.float32) 

# _ = play_game(actions)
next_emb, act_hist, emb_hist, frames, titles = play_game_v2(
    actions_5x2=actions,
    cost_model=cost_model,
    decoder=decoder_mc,
    adapter=adapter,
    scaler=action_scaler,
    act_hist=act_hist,
    emb_hist=emb_hist,
    frames=frames,
    titles=titles,
    device=DEVICE
)

In [ ]:
len(frames)

In [ ]:
frames[1].shape

In [ ]:
img_r=recolor_pusht(np.round(frames[1]*255).astype('uint8'), palette=DECODED_PALETTE, sigma=45) 
plt.imshow(img_r)

In [ ]:
Image.fromarray(img_r).save(save_dir+'/move_left.png')

In [ ]:
emb_hist_np=np.array([o.cpu().numpy().ravel() for o in emb_hist])
np.save(save_dir+'/move_left', emb_hist_np)

In [ ]:
emb_hist_np.shape

## Move Right

In [ ]:
# fresh env just to get a deterministic starting frame matched to Part A's seed=0 reset.
env_game = gym.make("swm/PushT-v1", render_mode="rgb_array")
env_game.reset(seed=2)
start_frame = env_game.render()
env_game.close()

# Initialise histories with the first observed frame
first_emb = encode_frame(start_frame, cost_model=cost_model, transform=img_transform, device=DEVICE)  # (1, D)
emb_hist = [first_emb]
act_hist = []

# Seed frames with the real observation for visual reference
frames = [((start_frame - start_frame.min()) / (start_frame.max() - start_frame.min() + 1e-8))]
titles = ["t=0 (real)"]

show_strip(frames, titles)

In [ ]:
actions = np.array([
    [0.2, 0.0],
    [0.2, 0.0],
    [0.2, 0.0],
    [0.2, 0.0],
    [0.2, 0.0], 
], dtype=np.float32) 

# _ = play_game(actions)
next_emb, act_hist, emb_hist, frames, titles = play_game_v2(
    actions_5x2=actions,
    cost_model=cost_model,
    decoder=decoder_mc,
    adapter=adapter,
    scaler=action_scaler,
    act_hist=act_hist,
    emb_hist=emb_hist,
    frames=frames,
    titles=titles,
    device=DEVICE
)

In [ ]:
img_r=recolor_pusht(np.round(frames[1]*255).astype('uint8'), palette=DECODED_PALETTE, sigma=45) 
Image.fromarray(img_r).save(save_dir+'/move_right.png')
plt.imshow(img_r)

In [ ]:
emb_hist_np=np.array([o.cpu().numpy().ravel() for o in emb_hist])
np.save(save_dir+'/move_right', emb_hist_np)

## Move Up

In [ ]:
# fresh env just to get a deterministic starting frame matched to Part A's seed=0 reset.
env_game = gym.make("swm/PushT-v1", render_mode="rgb_array")
env_game.reset(seed=2)
start_frame = env_game.render()
env_game.close()

# Initialise histories with the first observed frame
first_emb = encode_frame(start_frame, cost_model=cost_model, transform=img_transform, device=DEVICE)  # (1, D)
emb_hist = [first_emb]
act_hist = []

# Seed frames with the real observation for visual reference
frames = [((start_frame - start_frame.min()) / (start_frame.max() - start_frame.min() + 1e-8))]
titles = ["t=0 (real)"]

show_strip(frames, titles)

In [ ]:
actions = np.array([
    [0.0, -0.2],
    [0.0, -0.2],
    [0.0, -0.2],
    [0.0, -0.2],
    [0.0, -0.2], 
], dtype=np.float32) 

# _ = play_game(actions)
next_emb, act_hist, emb_hist, frames, titles = play_game_v2(
    actions_5x2=actions,
    cost_model=cost_model,
    decoder=decoder_mc,
    adapter=adapter,
    scaler=action_scaler,
    act_hist=act_hist,
    emb_hist=emb_hist,
    frames=frames,
    titles=titles,
    device=DEVICE
)

In [ ]:
img_r=recolor_pusht(np.round(frames[1]*255).astype('uint8'), palette=DECODED_PALETTE, sigma=45) 
Image.fromarray(img_r).save(save_dir+'/move_up.png')
plt.imshow(img_r)

Hmm why is my color mapping not working on up?

In [ ]:
emb_hist_np=np.array([o.cpu().numpy().ravel() for o in emb_hist])
np.save(save_dir+'/move_up', emb_hist_np)

## Move Down

In [ ]:
# fresh env just to get a deterministic starting frame matched to Part A's seed=0 reset.
env_game = gym.make("swm/PushT-v1", render_mode="rgb_array")
env_game.reset(seed=2)
start_frame = env_game.render()
env_game.close()

# Initialise histories with the first observed frame
first_emb = encode_frame(start_frame, cost_model=cost_model, transform=img_transform, device=DEVICE)  # (1, D)
emb_hist = [first_emb]
act_hist = []

# Seed frames with the real observation for visual reference
frames = [((start_frame - start_frame.min()) / (start_frame.max() - start_frame.min() + 1e-8))]
titles = ["t=0 (real)"]

show_strip(frames, titles)

In [ ]:
actions = np.array([
    [0.0, 0.15],
    [0.0, 0.15],
    [0.0, 0.15],
    [0.0, 0.15],
    [0.0, 0.15], 
], dtype=np.float32) 

# _ = play_game(actions)
next_emb, act_hist, emb_hist, frames, titles = play_game_v2(
    actions_5x2=actions,
    cost_model=cost_model,
    decoder=decoder_mc,
    adapter=adapter,
    scaler=action_scaler,
    act_hist=act_hist,
    emb_hist=emb_hist,
    frames=frames,
    titles=titles,
    device=DEVICE
)

In [ ]:
img_r=recolor_pusht(np.round(frames[1]*255).astype('uint8'), palette=DECODED_PALETTE, sigma=45) 
Image.fromarray(img_r).save(save_dir+'/move_down.png')
plt.imshow(img_r)

In [ ]:
emb_hist_np=np.array([o.cpu().numpy().ravel() for o in emb_hist])
np.save(save_dir+'/move_down', emb_hist_np)

## P71

- Ok I think it makes sense to stick with the same notebook for now!
- Ok so the whole thing here is comparing real rollouts to world model ones
- This section is probably a pretty good fit for Claude
- Due to action blocking, I think I can only show every 5th step - which is fine
- So I think for p71, I use the same expert trajectories
- And for both the simulated and real environments, for each expert trajectory I need to render: starting frame, every 5th frame, save all actions as npz, save all embeddings as npz. 

In [ ]:
DEFAULT_PALETTE_2: dict = {
    # ── source anchors (sampled by hand from the PushT renderer) ─────────
    "src_bg":            np.array([255, 255, 255], dtype=np.float32),
    "src_frame_border":  np.array([205, 205, 205], dtype=np.float32),
    "src_target_fill":   np.array([130, 236, 138], dtype=np.float32),  # green T
    "src_movable_fill":  np.array([130, 153, 172], dtype=np.float32),  # blue-gray T
    "src_effector_fill": np.array([ 70, 119, 246], dtype=np.float32),  # blue dot
    "src_target_outline":   np.array([170, 212, 161], dtype=np.float32),  # green T
    "src_movable_outline":  np.array([125, 136, 143], dtype=np.float32),  # blue-gray T
    "src_effector_outline": np.array([94, 115, 167], dtype=np.float32),  # blue dot
 
    # ── target colors ────────────────────────────────────────────────────
    "tgt_bg":               BLACK,
    "tgt_frame_border":     BLACK,
    "tgt_target_fill":      CHILL_BROWN,
    "tgt_target_outline":   CHILL_BROWN, # * 0.55,
    "tgt_movable_fill":     BLUE,
    "tgt_movable_outline":  BLUE * 0.75,
    "tgt_effector_fill":    YELLOW,
    "tgt_effector_outline": YELLOW,
 
    # ── fill-mask thresholds (loose color-ball radii) ────────────────────
    # Only used to seed the Voronoi label propagation; doesn't need to be tight.
    "r_target_fill":   45.0,
    "r_movable_fill":  22.0,
    "r_effector_fill": 55.0,
 
    # ── connected-component cleanup ──────────────────────────────────────
    # Drop fill-mask components smaller than this many pixels (kills stray AA
    # pixels that happen to fall inside a fill color ball).
    "min_component_size": 10,
}

In [ ]:
from pathlib import Path
from PIL import Image

EP_ID = 18002   # change this to pick a different human episode, ok let's try this one!

OUT_ROOT = Path('data/hackin/p71')

# Episode boundaries from cached lengths
ep_lens   = np.asarray(ds.lengths, dtype=np.int64)
ep_starts = np.concatenate([[0], np.cumsum(ep_lens)[:-1]])
ep_len    = int(ep_lens[EP_ID])
row0      = int(ep_starts[EP_ID])

ep_dir   = OUT_ROOT / f"ep_{EP_ID:04d}"
real_dir = ep_dir / "real"
wm_dir   = ep_dir / "wm"
real_dir.mkdir(parents=True, exist_ok=True)
wm_dir.mkdir(parents=True, exist_ok=True)

# Pull actions + initial state from the dataset (both are cached)
actions_env = np.stack([
    ds[row0 + t]["action"].squeeze(0).cpu().numpy() for t in range(ep_len)
]).astype(np.float32)                                                  # (ep_len, 2) env-scale
init_state  = ds[row0]["state"].squeeze(0).cpu().numpy().astype(np.float32)  # (7,)

print(f"Episode {EP_ID}: rows {row0}..{row0+ep_len-1}, length {ep_len}")
print(f"actions_env: {actions_env.shape}   init_state: {init_state.round(3)}")
print(f"→ real dir: {real_dir}")
print(f"→ wm   dir: {wm_dir}")

In [ ]:
env_play = gym.make("swm/PushT-v1", render_mode="rgb_array",
                    max_episode_steps=ep_len + 10)
env_play.reset(seed=0)
env_play.unwrapped._set_state(state=init_state)

# t = 0
frame_t = env_play.render()                                            # (H,W,3) uint8
emb_t   = encode_frame(frame_t, cost_model=cost_model,
                       transform=img_transform, device=DEVICE)         # (1, D)

real_embs        = [emb_t.cpu().numpy().ravel()]
saved_real_steps = [0]

img_r = recolor_pusht(frame_t, palette=DEFAULT_PALETTE_2, sigma=30)
Image.fromarray(img_r).save(real_dir / f"frame_{0:04d}.png")

# Stash for the WM cell so we don't double-render
start_frame_real = frame_t.copy()
first_emb_real   = emb_t

for t in range(ep_len):
    obs, _, term, trunc, _ = env_play.step(actions_env[t])
    frame_t = env_play.render()
    emb_t   = encode_frame(frame_t, cost_model=cost_model,
                           transform=img_transform, device=DEVICE)
    real_embs.append(emb_t.cpu().numpy().ravel())

    step_idx = t + 1
    if step_idx % 5 == 0:
        img_r = recolor_pusht(frame_t, palette=DEFAULT_PALETTE_2, sigma=30)
        Image.fromarray(img_r).save(real_dir / f"frame_{step_idx:04d}.png")
        saved_real_steps.append(step_idx)


    if term or trunc:
        print(f"env ended at t={step_idx}  (term={term}, trunc={trunc})")
        break

env_play.close()

real_embs = np.stack(real_embs, axis=0)   # (ep_len+1, D)  one emb per env step
np.savez(
    real_dir / "data.npz",
    actions      = actions_env,            # (ep_len, 2) env-scale
    embeddings   = real_embs,              # (ep_len+1, D)  D = 192
    saved_steps  = np.asarray(saved_real_steps, dtype=np.int64),
    init_state   = init_state,
    episode_idx  = np.int64(EP_ID),
)
print(f"real rollout: {len(saved_real_steps)} png  |  embeddings {real_embs.shape}")

In [ ]:
Image.fromarray(img_r)

In [ ]:
    # "src_bg":            np.array([254, 254, 254], dtype=np.float32),
    # "src_frame_border":  np.array([197,	197,	197		], dtype=np.float32),
    # "src_target_fill":   np.array([91,	229,	104	], dtype=np.float32),  # green T
    # "src_movable_fill":  np.array([92,	121,	147	], dtype=np.float32),  # blue-gray T
    # "src_effector_fill": np.array([11,	78,	248	], dtype=np.float32),  # blue dot
    # "src_target_outline":   np.array([104,	231,	115	], dtype=np.float32),  # green T
    # "src_movable_outline":  np.array([64,	85,	106	], dtype=np.float32),  # blue-gray T
    # "src_effector_outline": np.array([11,	65,	212	], dtype=np.float32),  # blue dot

### "Video Game" test on this episode before running

In [ ]:
emb_hist = [first_emb_real]
act_hist = []

wm_embs        = [first_emb_real.cpu().numpy().ravel()]
saved_wm_steps = [0]

img0 = decode_emb(first_emb_real, adapter, decoder_mc)
img_r = recolor_pusht(np.round(img0 * 255).astype('uint8'),
                      palette=DECODED_PALETTE, sigma=45)
Image.fromarray(img_r).save(ep_dir / "before.png")

In [ ]:
emb_hist = [first_emb_real]
act_hist = []

wm_embs        = [first_emb_real.cpu().numpy().ravel()]
saved_wm_steps = [0]


a5 = np.array([
    [-0.15, 0.0],
    [-0.15, 0.0],
    [-0.15, 0.0],
    [-0.15, 0.0],
    [-0.15, 0.0], #LEFT
], dtype=np.float32) 

a_block = pack_action_block(a5, action_scaler, DEVICE)              # (1, 10) standardised
act_hist.append(a_block)

next_emb = predict_next(cost_model, emb_hist, act_hist, history_size=3)  # (1, D)
emb_hist.append(next_emb)

img   = decode_emb(next_emb, adapter, decoder_mc)
img_r = recolor_pusht(np.round(img * 255).astype('uint8'),
                      palette=DECODED_PALETTE, sigma=45)
Image.fromarray(img_r).save(ep_dir / "left.png")

In [ ]:
Image.fromarray(img_r)

In [ ]:
emb_hist = [first_emb_real]
act_hist = []

wm_embs        = [first_emb_real.cpu().numpy().ravel()]
saved_wm_steps = [0]


a5 = np.array([
    [0.15, 0.0],
    [0.15, 0.0],
    [0.15, 0.0],
    [0.15, 0.0],
    [0.15, 0.0], #LEFT
], dtype=np.float32) 

a_block = pack_action_block(a5, action_scaler, DEVICE)              # (1, 10) standardised
act_hist.append(a_block)

next_emb = predict_next(cost_model, emb_hist, act_hist, history_size=3)  # (1, D)
emb_hist.append(next_emb)

img   = decode_emb(next_emb, adapter, decoder_mc)
img_r = recolor_pusht(np.round(img * 255).astype('uint8'),
                      palette=DECODED_PALETTE, sigma=45)
Image.fromarray(img_r).save(ep_dir / "right.png")

In [ ]:
emb_hist = [first_emb_real]
act_hist = []

wm_embs        = [first_emb_real.cpu().numpy().ravel()]
saved_wm_steps = [0]


a5 = np.array([
    [0.0, 0.15],
    [0.0, 0.15],
    [0.0, 0.15],
    [0.0, 0.15],
    [0.0, 0.15], #DOWN
], dtype=np.float32) 

a_block = pack_action_block(a5, action_scaler, DEVICE)              # (1, 10) standardised
act_hist.append(a_block)

next_emb = predict_next(cost_model, emb_hist, act_hist, history_size=3)  # (1, D)
emb_hist.append(next_emb)

img   = decode_emb(next_emb, adapter, decoder_mc)
img_r = recolor_pusht(np.round(img * 255).astype('uint8'),
                      palette=DECODED_PALETTE, sigma=45)
Image.fromarray(img_r).save(ep_dir / "down.png")

In [ ]:
emb_hist = [first_emb_real]
act_hist = []

wm_embs        = [first_emb_real.cpu().numpy().ravel()]
saved_wm_steps = [0]


a5 = np.array([
    [0.0, -0.15],
    [0.0, -0.15],
    [0.0, -0.15],
    [0.0, -0.15],
    [0.0, -0.15], #DOWN
], dtype=np.float32) 

a_block = pack_action_block(a5, action_scaler, DEVICE)              # (1, 10) standardised
act_hist.append(a_block)

next_emb = predict_next(cost_model, emb_hist, act_hist, history_size=3)  # (1, D)
emb_hist.append(next_emb)

img   = decode_emb(next_emb, adapter, decoder_mc)
img_r = recolor_pusht(np.round(img * 255).astype('uint8'),
                      palette=DECODED_PALETTE, sigma=45)
Image.fromarray(img_r).save(ep_dir / "up.png")

In [ ]:
# n_blocks = how many 5-action chunks fit in the episode (we truncate the tail)
n_blocks = ep_len // 5

emb_hist = [first_emb_real]
act_hist = []

wm_embs        = [first_emb_real.cpu().numpy().ravel()]
saved_wm_steps = [0]

# t=0: decoded version of the encoded real start (a gut-check on the adapter+decoder).
# If you'd rather make t=0 visually identical to the real rollout, replace this
# block with the same recolor_pusht(start_frame_real, palette=DEFAULT_PALETTE,...)
# save from Cell 2.
img0 = decode_emb(first_emb_real, adapter, decoder_mc)
img_r = recolor_pusht(np.round(img0 * 255).astype('uint8'),
                      palette=DECODED_PALETTE, sigma=45)
Image.fromarray(img_r).save(wm_dir / f"frame_{0:04d}.png")

for k in range(n_blocks):
    a5      = actions_env[k*5:(k+1)*5]                                  # (5, 2) env-scale
    a_block = pack_action_block(a5, action_scaler, DEVICE)              # (1, 10) standardised
    act_hist.append(a_block)

    next_emb = predict_next(cost_model, emb_hist, act_hist, history_size=3)  # (1, D)
    emb_hist.append(next_emb)

    img   = decode_emb(next_emb, adapter, decoder_mc)
    img_r = recolor_pusht(np.round(img * 255).astype('uint8'),
                          palette=DECODED_PALETTE, sigma=45)

    step_idx = (k + 1) * 5
    Image.fromarray(img_r).save(wm_dir / f"frame_{step_idx:04d}.png")

    wm_embs.append(next_emb.cpu().numpy().ravel())
    saved_wm_steps.append(step_idx)

wm_embs           = np.stack(wm_embs, axis=0)                                    # (n_blocks+1, D)
action_blocks_std = np.stack([a.cpu().numpy().ravel() for a in act_hist], axis=0) # (n_blocks, 10)
np.savez(
    wm_dir / "data.npz",
    embeddings    = wm_embs,
    action_blocks = action_blocks_std,     # standardised 10-d blocks the WM actually saw
    saved_steps   = np.asarray(saved_wm_steps, dtype=np.int64),
    episode_idx   = np.int64(EP_ID),
)
print(f"wm rollout:   {len(saved_wm_steps)} png  |  embeddings {wm_embs.shape}  "
      f"|  n_blocks={n_blocks} (last frame at t={n_blocks*5})")

In [ ]:
plt.imshow(img_r)

Ok I think this is making sense! Now, let me iterate over a bunch of random samples and I can pick ones I like!

In [ ]:
from tqdm import tqdm

In [ ]:
np.random.seed(25)
rI = np.arange(len(ep_lens))
np.random.shuffle(rI)

In [ ]:
for count in tqdm(range(100)):

    EP_ID = rI[count]   # change this to pick a different human episode
    
    OUT_ROOT = Path('data/hackin/p71b')
    
    # Episode boundaries from cached lengths
    ep_lens   = np.asarray(ds.lengths, dtype=np.int64)
    ep_starts = np.concatenate([[0], np.cumsum(ep_lens)[:-1]])
    ep_len    = int(ep_lens[EP_ID])
    row0      = int(ep_starts[EP_ID])
    
    ep_dir   = OUT_ROOT / f"ep_{EP_ID:04d}"
    real_dir = ep_dir / "real"
    wm_dir   = ep_dir / "wm"
    real_dir.mkdir(parents=True, exist_ok=True)
    wm_dir.mkdir(parents=True, exist_ok=True)
    
    # Pull actions + initial state from the dataset (both are cached)
    actions_env = np.stack([
        ds[row0 + t]["action"].squeeze(0).cpu().numpy() for t in range(ep_len)
    ]).astype(np.float32)                                                  # (ep_len, 2) env-scale
    init_state  = ds[row0]["state"].squeeze(0).cpu().numpy().astype(np.float32)  # (7,)
    
    print(f"Episode {EP_ID}: rows {row0}..{row0+ep_len-1}, length {ep_len}")
    print(f"actions_env: {actions_env.shape}   init_state: {init_state.round(3)}")
    print(f"→ real dir: {real_dir}")

    
    env_play = gym.make("swm/PushT-v1", render_mode="rgb_array",
                    max_episode_steps=ep_len + 10)
    env_play.reset(seed=0)
    env_play.unwrapped._set_state(state=init_state)
    
    # t = 0
    frame_t = env_play.render()                                            # (H,W,3) uint8
    emb_t   = encode_frame(frame_t, cost_model=cost_model,
                           transform=img_transform, device=DEVICE)         # (1, D)
    
    real_embs        = [emb_t.cpu().numpy().ravel()]
    saved_real_steps = [0]
    
    img_r = recolor_pusht(frame_t, palette=DEFAULT_PALETTE_2, sigma=30)
    Image.fromarray(img_r).save(real_dir / f"frame_{0:04d}.png")
    
    # Stash for the WM cell so we don't double-render
    start_frame_real = frame_t.copy()
    first_emb_real   = emb_t
    
    for t in range(ep_len):
        obs, _, term, trunc, _ = env_play.step(actions_env[t])
        frame_t = env_play.render()
        emb_t   = encode_frame(frame_t, cost_model=cost_model,
                               transform=img_transform, device=DEVICE)
        real_embs.append(emb_t.cpu().numpy().ravel())
    
        step_idx = t + 1
        if step_idx % 5 == 0:
            img_r = recolor_pusht(frame_t, palette=DEFAULT_PALETTE_2, sigma=30)
            Image.fromarray(img_r).save(real_dir / f"frame_{t:04d}.png")
            saved_real_steps.append(step_idx)
    
        if term or trunc:
            print(f"env ended at t={step_idx}  (term={term}, trunc={trunc})")
            break
    
    env_play.close()
    
    real_embs = np.stack(real_embs, axis=0)   # (ep_len+1, D)  one emb per env step
    np.savez(
        real_dir / "data.npz",
        actions      = actions_env,            # (ep_len, 2) env-scale
        embeddings   = real_embs,              # (ep_len+1, D)  D = 192
        saved_steps  = np.asarray(saved_real_steps, dtype=np.int64),
        init_state   = init_state,
        episode_idx  = np.int64(EP_ID),
    )
    print(f"real rollout: {len(saved_real_steps)} png  |  embeddings {real_embs.shape}")


    # n_blocks = how many 5-action chunks fit in the episode (we truncate the tail)
    n_blocks = ep_len // 5
    
    emb_hist = [first_emb_real]
    act_hist = []
    
    wm_embs        = [first_emb_real.cpu().numpy().ravel()]
    saved_wm_steps = [0]
    
    # t=0: decoded version of the encoded real start (a gut-check on the adapter+decoder).
    # If you'd rather make t=0 visually identical to the real rollout, replace this
    # block with the same recolor_pusht(start_frame_real, palette=DEFAULT_PALETTE,...)
    # save from Cell 2.
    img0 = decode_emb(first_emb_real, adapter, decoder_mc)
    img_r = recolor_pusht(np.round(img0 * 255).astype('uint8'),
                          palette=DECODED_PALETTE, sigma=45)
    Image.fromarray(img_r).save(wm_dir / f"frame_{0:04d}.png")
    
    for k in range(n_blocks):
        a5      = actions_env[k*5:(k+1)*5]                                  # (5, 2) env-scale
        a_block = pack_action_block(a5, action_scaler, DEVICE)              # (1, 10) standardised
        act_hist.append(a_block)
    
        next_emb = predict_next(cost_model, emb_hist, act_hist, history_size=3)  # (1, D)
        emb_hist.append(next_emb)
    
        img   = decode_emb(next_emb, adapter, decoder_mc)
        img_r = recolor_pusht(np.round(img * 255).astype('uint8'),
                              palette=DECODED_PALETTE, sigma=45)
    
        step_idx = (k + 1) * 5
        Image.fromarray(img_r).save(wm_dir / f"frame_{k:04d}.png")
    
        wm_embs.append(next_emb.cpu().numpy().ravel())
        saved_wm_steps.append(step_idx)
    
    wm_embs           = np.stack(wm_embs, axis=0)                                    # (n_blocks+1, D)
    action_blocks_std = np.stack([a.cpu().numpy().ravel() for a in act_hist], axis=0) # (n_blocks, 10)
    np.savez(
        wm_dir / "data.npz",
        embeddings    = wm_embs,
        action_blocks = action_blocks_std,     # standardised 10-d blocks the WM actually saw
        saved_steps   = np.asarray(saved_wm_steps, dtype=np.int64),
        episode_idx   = np.int64(EP_ID),
    )
    print(f"wm rollout:   {len(saved_wm_steps)} png  |  embeddings {wm_embs.shape}  "
          f"|  n_blocks={n_blocks} (last frame at t={n_blocks*5})")

---

In [ ]:
len(emb_hist)

In [ ]:
emb_hist[0].shape

In [ ]:
emb_hist[1].shape

In [ ]:
act_hist

In [ ]:
next_emb.shape

In [ ]:
plt.plot(emb_hist[0].detach().cpu().numpy().ravel())
plt.plot(emb_hist[1].detach().cpu().numpy().ravel())
plt.plot(next_emb.detach().cpu().numpy().ravel())

In [ ]:
# cost_model